In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!ls

Cap3_MedicionPobreza  drive  sample_data


In [2]:
!rm -rf Cap3_MedicionPobreza
!git clone https://github.com/LizamaD/Cap3_MedicionPobreza.git

Cloning into 'Cap3_MedicionPobreza'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 43 (delta 13), reused 42 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 25.72 KiB | 25.72 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [3]:
import pandas as pd

path_bases = "/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/"
path_pobreza = '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Base final/pobreza_24.csv'


# --- Carga tus datos crudos ---
poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")
trabajos_raw = pd.read_csv(f"{path_bases}trabajos.csv")
gastospersona_raw = pd.read_csv(f"{path_bases}gastospersona.csv")
gastoshogar_raw = pd.read_csv(f"{path_bases}gastoshogar.csv")
ingresos_raw = pd.read_csv(f"{path_bases}ingresos.csv")
pobreza_raw = pd.read_csv(path_pobreza)

/tmp/ipykernel_8214/1323284465.py:8: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
/tmp/ipykernel_8214/1323284465.py:9: DtypeWarning: Columns (1,4,27,44) have mixed types. Specify dtype option on import or set low_memory=False.
  viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
/tmp/ipykernel_8214/1323284465.py:10: DtypeWarning: Columns (45,49,53,61,63,69,75,77,81,83,85) have mixed types. Specify dtype option on import or set low_memory=False.
  hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")


In [4]:
%cd /content/Cap3_MedicionPobreza

/content/Cap3_MedicionPobreza


In [5]:
from src.trabajos import process_trabajos
from src.ingresos import generar_ingreso_deflactado_ago2024
from src.gastoshogar import procesar_gastos_enigh
from src.gastospersona import procesar_gastos_persona_enigh
from src.hogares import process_hogares
from src.poblacion import process_poblacion
from src.viviendas import process_viviendas
from src.pipeline import create_master_table, impute_data

In [10]:
pob_keys = pobreza_raw[['folioviv','foliohog','numren','pobreza','pobreza_e']]
pob_proc = process_poblacion(poblacion_raw)
pob_keys.merge(pob_proc, on=['folioviv', 'foliohog', 'numren'], how='left').head(2)

ValueError: You are trying to merge on int64 and object columns for key 'folioviv'. If you wish to proceed you should use pd.concat

In [6]:
# --- Crea la tabla maestra ---
# Esta función procesa, une todo y guarda el resultado en "master_table.csv"
master_df = create_master_table(
    pob_keys=pobreza_raw,
    pob_df=poblacion_raw,
    viv_df=viviendas_raw,
    hog_df=hogares_raw,
    trab_df=trabajos_raw,
    gasper_df=gastospersona_raw,
    gashog_df=gastoshogar_raw,
    ing_df=ingresos_raw,
    output_path="data/processed/enigh_master_table.csv" # Ruta sugerida
)

# --- Imputa los datos faltantes ---
# Esta función toma la tabla maestra y la deja lista para el modelado
df_final_limpio = impute_data(master_df)

# --- Verifica el resultado ---
print("\nDataFrame final después de la imputación:")
print(df_final_limpio.head())

# Confirma que no hay valores nulos
print("\nSuma de nulos por columna en el DF final:")
print(df_final_limpio.isnull().sum().sum()) # Debería ser 0


Procesando tablas individuales...


/content/Cap3_MedicionPobreza/src/ingresos.py:34: FutureWarning: The provided callable <function sum at 0x7885607b8360> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  aguinaldo = pd.pivot_table(


Uniendo tablas en una tabla maestra...


ValueError: You are trying to merge on int64 and object columns for key 'folioviv'. If you wish to proceed you should use pd.concat

In [7]:
master_df.head(2)

NameError: name 'master_df' is not defined